## Installs

In [9]:
RUNINSTALLS = False

if RUNINSTALLS:
  !pip install openpyxl
  !pip install --upgrade google-cloud-aiplatform


## Notebook Setup

In [10]:
from IPython.display import HTML, display

def set_css(arg=None):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

import logging
import sys
format_string = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
logger = logging.getLogger()
fhandler = logging.FileHandler(filename='notebook.log', mode='a')
formatter = logging.Formatter(format_string)
fhandler.setFormatter(formatter)
#logger.addHandler(fhandler)  #uncomment if you want a log file
logging.basicConfig(format=format_string,
                     level=logging.INFO, stream=sys.stdout)
logger.setLevel(logging.INFO)


## Imports

In [11]:
from io import BytesIO
import datetime
import yaml
import json
import pandas as pd
import openpyxl
import vertexai
from google.cloud import storage
from vertexai.generative_models import GenerationConfig, GenerativeModel, Part
from vertexai.preview import caching

## Project setup

In [12]:
BUCKET_NAME = "uk-bh-experiments-argolis-us"
FOLDER_PATH = "subsea7/hseq_data/"
MODEL_NAME = "gemini-1.5-pro-001"

vertexai.init(project="uk-bh-experiments-argolis", location="us-central1")

In [13]:

def get_description_column(df):
  column_names = list(df)
  for column_name in column_names:
      if "desc" in column_name.lower():
         return column_name
  #if none found, return the 4th column
  return column_names[3]


def get_excel_sheet( file_name, sheet_name=None, header=0, mandatory_columns=None):

  storage_client = storage.Client()
  bucket = storage_client.bucket(BUCKET_NAME)
  blob = bucket.blob(file_name)
  logging.info(f"Got file {file_name}")

  with blob.open("rb") as f:
      file_bytes = BytesIO(f.read())

  logging.info(f"read file {file_name}")

  openpyxl.reader.excel.warnings.simplefilter(action='ignore')
  gs_uri = f"gs://{BUCKET_NAME}/{FOLDER_PATH}/{file_name}"

  if sheet_name is None:
    #xls = pd.ExcelFile(gs_uri,sheetname=None, nrows=0, engine='openpyxl')
    sheet_names = pd.ExcelFile(file_bytes,  engine='openpyxl').sheet_names
    print(f"Available sheets:")
    for sheet_name  in sheet_names:
        print(f"{sheet_name}")
    return sheet_names
  else:
    print(f"Reading sheet {sheet_name}")

    with pd.ExcelFile(file_bytes) as xls:
      df = pd.read_excel(xls, sheet_name, header=header)
      print(f"Sheet {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      if mandatory_columns == None:
         mandatory_columns = [get_description_column(df)]
      logging.info(f"Dropping all rows that have nothing in the columns: {mandatory_columns}")
      df.dropna(subset=mandatory_columns, inplace=True)
      print(f"Cleaned Rows {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      logging.info("Dropping columns that have have nothing in any rows")
      df.dropna(axis=1, how="all", inplace = True)

      print(f"Cleaned Columns {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")

    return df


def df_to_json(df):
  sheet_json_str = df.to_json(orient='records')
  #print(f'{sheet_json_str[:1]}')
  sheet_json = json.loads(sheet_json_str)
  return sheet_json_str


def df_select_columns(df):
   print(df.head())
   print(f"{list(df)}")

## Add Caching



In [14]:
def cache_document(documents):

    system_instruction = """
    You are an safety expert. You always stick to the facts in the sources provided, and never make up new facts.
    Now look at these lists of saftey observations, and answer the following questions.
    """

    contents = [Part.from_text(document) for document in documents]

    cached_content = caching.CachedContent.create(
        model_name=MODEL_NAME,
        system_instruction=system_instruction,
        contents=contents,
        ttl=datetime.timedelta(minutes=60),
    )

    print(cached_content.name)
    return cached_content.name


def generate_from_cache(cache_id, question):


    cached_content = caching.CachedContent(cached_content_name=cache_id)

    model = GenerativeModel.from_cached_content(cached_content=cached_content)

    response = model.generate_content(question)

    return response.text


## Use JSON as TXT

In [27]:
def generate_using_text(documents, question):
    
    system_instruction = """
    You are an safety expert. You always stick to the facts in the sources provided, and never make up new facts.
    Now look at these lists of saftey observations, and answer the following questions.
    """

    prompt = f'''
        <task> Provide a helpful and factual answer to the question that the user has asked</task>
        <question>{question}</question>
        <output>As well as giving a summary answer to the question, provide 3-6 examples of obvervations from the documents that support your conclusion</output>
    '''

    model = GenerativeModel(MODEL_NAME)
    model = GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=[
        system_instruction,
    ],
)
    generation_config=GenerationConfig(
        temperature = 0.8
    )

    contents = [Part.from_text(document) for document in documents]
    contents.append(prompt)
    response = model.count_tokens(contents)
    logging.debug(f"Prompt Token Count: {response.total_tokens}")
    logging.debug(f"Prompt Character Count: {response.total_billable_characters}")

    response = model.generate_content(contents,generation_config=generation_config,stream=False)

    # Response tokens count
    usage_metadata = response.usage_metadata
    logging.info(f"Response Prompt Token Count: {usage_metadata.prompt_token_count}")
    logging.info(f"Candidates Token Count: {usage_metadata.candidates_token_count}")
    logging.info(f"Total Token Count: {usage_metadata.total_token_count}")

    response_text = response.text

    return response_text


In [16]:
df2 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
df2.head()
df_select_columns(df2)


2024-07-12 12:56:35,968 - root - INFO - Got file subsea7/hseq_data/NormandSubsea.xlsx
2024-07-12 12:56:36,672 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
2024-07-12 12:56:38,712 - root - INFO - Dropping all rows that have nothing in the columns: ['Description of observation']
Cleaned Rows OBSERVATIONS has 34 rows and 61 columns
2024-07-12 12:56:38,716 - root - INFO - Dropping columns that have have nothing in any rows
Cleaned Columns OBSERVATIONS has 34 rows and 35 columns
   Obs. No                 Date Name of observer       Department  \
0    120.0  2021-01-02 00:00:00    Brian Bullock   Project - Deck   
1    121.0  2021-03-01 00:00:00     A. Greenwood          Tooling   
2    122.0  2021-03-01 00:00:00     A. Greenwood          Tooling   
3    123.0  2021-03-01 00:00:00     Oysten Bauge  Marine - Engine   
4    124.0  2021-03-02 00:00:00     A. Greenwood          Tooling   

              

In [17]:
sheet_json_str = df2.to_json(orient='records')
observation = generate_using_text([sheet_json_str], "what is the most unsafe location")
print(f'{observation}')

I0000 00:00:1720785399.728893   32586 config.cc:230] gRPC experiments enabled: call_status_override_on_cancellation, event_engine_dns, event_engine_listener, http2_stats_fix, monitoring_experiment, pick_first_new, trace_record_callops, work_serializer_clears_time_cache, work_serializer_dispatch
I0000 00:00:1720785399.729633   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 71


Prompt Token Count: 11053
Prompt Character Count: 40672
Response Prompt Token Count: 11099
Candidates Token Count: 179
Total Token Count: 11278
Based on the provided safety observations, there isn't enough information to determine the most unsafe *location*. The observations highlight various safety concerns in different areas and departments. For example:

* **Observation 123** points out a potential slipping hazard due to a water leak in a public toilet on the 1st deck.
* **Observation 126** mentions a broken fluorescent light bulb presenting a hazard in the online room.
* **Observation 132** describes a loose light fixture and corroded bolts in the hangar, posing risks of falling objects.
* **Observation 149** reports a drain issue in the hospital bathroom causing flooding and rendering the sink unusable.

These are just a few examples, and it's impossible to pinpoint one "most unsafe" location without further context or a comprehensive analysis of all areas on board. 



I0000 00:00:1720785406.396453     310 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720785406.398840     310 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [18]:
df1 = get_excel_sheet("subsea7/hseq_data/SevenArctic.xlsx", "OBS", 2)
df2 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
df3 = get_excel_sheet(f"{FOLDER_PATH}SeawayStrashnov.xlsx", "Obs_Int Register", 1)

docs = [df_to_json(df1), df_to_json(df2), df_to_json(df3)]


2024-07-12 12:56:48,140 - root - INFO - Got file subsea7/hseq_data/SevenArctic.xlsx
2024-07-12 12:56:49,636 - root - INFO - read file subsea7/hseq_data/SevenArctic.xlsx
Reading sheet OBS
Sheet OBS has 7538 rows and 19 columns
2024-07-12 12:56:59,282 - root - INFO - Dropping all rows that have nothing in the columns: ['Description']
Cleaned Rows OBS has 7507 rows and 19 columns
2024-07-12 12:56:59,290 - root - INFO - Dropping columns that have have nothing in any rows
Cleaned Columns OBS has 7507 rows and 19 columns
2024-07-12 12:57:01,164 - root - INFO - Got file subsea7/hseq_data/NormandSubsea.xlsx
2024-07-12 12:57:02,277 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
2024-07-12 12:57:04,724 - root - INFO - Dropping all rows that have nothing in the columns: ['Description of observation']
Cleaned Rows OBSERVATIONS has 34 rows and 61 columns
2024-07-12 12:57:04,729 - root - INFO - Dropping columns

I0000 00:00:1720785432.371547   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 75


Prompt Token Count: 1461472
Prompt Character Count: 4882815
Response Prompt Token Count: 1461518
Candidates Token Count: 38
Total Token Count: 1461556
This question cannot be answered from the given context. The provided data is a list of safety observations, but it doesn't contain an objective analysis to determine the most unsafe location. 



I0000 00:00:1720785534.047191     418 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720785534.048371     418 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


## Questions

Questions from John:
 - Where on our worksites is someone most likely to be hurt?  What were they doing?
 - What is the most likely way someone could be hurt?
 - Do we have a problem with doors?
 - If I was to tackle one issue on our worksites, what would it be?
 - Are our gallies safe?
 

Can AI be used on our Synergi, RA7 and MOC databases?
(Synergi is our HSEQ incident reporting tool, RA7 records risk assessments, MOC is our Management of Change tool)

 - When we change rigging offshore, do we normally increase or decrease the capacity?
 - Do we have a problem with dropped tools?

In [28]:
question = "Where on our worksites is someone most likely to be hurt?  What were they doing?"
observation = generate_using_text(docs, question)
print(f'{observation}')

I0000 00:00:1720785999.917556   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 70


Prompt Token Count: 1461485
Prompt Character Count: 4882853
Response Prompt Token Count: 1461531
Candidates Token Count: 46
Total Token Count: 1461577
This question cannot be answered from the given context. There are many observations on unsafe conditions and practices, but there is no information on where people are most likely to be hurt, or what they were doing at the time. 



I0000 00:00:1720786095.772475     702 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720786095.775226     702 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [20]:
question = "What is the most likely way someone could be hurt?"
observation = generate_using_text(docs, question)
print(f'{observation}')

I0000 00:00:1720785590.841512   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 75


Prompt Token Count: 1461477
Prompt Character Count: 4882829
Response Prompt Token Count: 1461523
Candidates Token Count: 218
Total Token Count: 1461741
Based on the safety observations provided, the most common hazard is a **slip, trip, or fall**. This is due to a variety of factors, including:

* **Grease on decks and stairwells:** Many observations report grease build-up on walking surfaces, especially around cranes and in workshops. 
* **Wet surfaces:** Spills, leaks, and condensation are frequently reported, creating slippery surfaces in accommodation areas, stairwells, and on deck.
* **Loose objects:** Tools, equipment, and debris are often found in walkways and on stairs, posing trip hazards.
* **Unsecured items:** Items stored on top of lockers, cabinets, and other surfaces can fall and cause injury. 
* **Inadequate barriers:** Missing or poorly placed barriers can lead to falls from heights or into open spaces like the moonpool.
* **Weathertight doors left open:** Swinging door

I0000 00:00:1720785636.854445     510 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720785636.855532     510 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [21]:
question = "Do we have a problem with doors?"
observation = generate_using_text(docs, question)
print(f'{observation}')

I0000 00:00:1720785636.880848   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 75


Prompt Token Count: 1461474
Prompt Character Count: 4882814
Response Prompt Token Count: 1461520
Candidates Token Count: 24
Total Token Count: 1461544
This safety data suggests that there are recurring issues with doors being left open and unsecured, particularly weathertight doors. 



I0000 00:00:1720785677.714944     536 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720785677.715764     536 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


In [22]:
question = "If I was to tackle one issue on our worksites, what would it be?"
observation = generate_using_text(docs, question)
print(f'{observation}')

I0000 00:00:1720785677.736333   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 75


Prompt Token Count: 1461483
Prompt Character Count: 4882839
Response Prompt Token Count: 1461529
Candidates Token Count: 50
Total Token Count: 1461579
Based on the provided data, the most frequent safety observation is "**Housekeeping**". There are numerous reports of unsecured items, spills, misplaced tools, and improper waste segregation. Addressing housekeeping issues would significantly improve safety on the worksites. 



I0000 00:00:1720785716.497120     549 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720785716.497552     549 work_stealing_thread_pool.cc:269] WorkStealingThreadPoolImpl::Quiesce


## Try as YAML

In [23]:

df_yaml = yaml.dump(df1.to_dict(orient='records'),default_flow_style=None)

In [24]:
print(f'{df_yaml[:1000]}')

- Action Undertaken: .nan
  ? 'Card

    ID'
  : 2
  Corrective/Immediate Action Taken: Informed bridge and bosun
  ? 'Date

    Closed DD/MM/YYYY'
  : .nan
  Day: 1.0
  Department: Marine Deck
  Description: Oxygen bottles stored in area labelled acetylene and vice versa
  Location: Main Deck
  Month: Feb
  Name of Observer: R. Cooper
  OBS: Obs
  OPEN / Closed: Closed
  ? 'Person Responsible

    for Action'
  : C/O
  Safety Improvement (Yes/No): .nan
  Suggested Further Action Required: Switch bottles or signage
  Time: &id031 !!python/object/apply:datetime.time
  - !!binary |
    CgAAAAAA
  'Unnamed: 17': .nan
  'Unnamed: 18': .nan
  Year: 2017
- Action Undertaken: .nan
  ? 'Card

    ID'
  : 3
  Corrective/Immediate Action Taken: Closed tap
  ? 'Date

    Closed DD/MM/YYYY'
  : .nan
  Day: 1.0
  Department: Marine Deck
  Description: 'Found tap in bridge cleaning locker not properly closed and dripping.
    Waste of water. '
  Location: Bridge
  Month: Feb
  Name of Observer: D. P

In [29]:
observation = generate_using_text(df_yaml, "what is the most unsafe location")
print(observation)

I0000 00:00:1720786208.554678   32586 ev_epoll1_linux.cc:125] grpc epoll fd: 68


Prompt Token Count: 5043514
Prompt Character Count: 4018620


ResourceExhausted: 429 Quota exceeded for aiplatform.googleapis.com/generate_content_input_tokens_per_minute_per_base_model with base model: gemini-1.5-pro. Please submit a quota increase request. https://cloud.google.com/vertex-ai/docs/generative-ai/quotas-genai.

In [ ]:
print(observation)